<a href="https://colab.research.google.com/github/Praharshita1275/Agentic-Medical-Fact-Verification-system/blob/main/MEMBER_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ersindemirel/pubhealthdataset")

print("Path to dataset files:", path)

100%|██████████| 21.6M/21.6M [00:00<00:00, 108MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ersindemirel/pubhealthdataset/versions/1


In [ ]:
import os

print(os.listdir(path))

['test.tsv', 'train.tsv']


In [ ]:
import pandas as pd
import os

train_path = os.path.join(path, 'train.tsv')

df = pd.read_csv(train_path, sep='\t')
df.head()

,claim_id,claim,date_published,explanation,fact_checkers,main_text,sources,label,subjects
0,15661,"""The money the Clinton Foundation took from fr...","April 26, 2015","""Gingrich said the Clinton Foundation """"took m...",Katie Sanders,"""Hillary Clinton is in the political crosshair...",https://www.wsj.com/articles/clinton-foundatio...,false,"Foreign Policy, PunditFact, Newt Gingrich,"
1,9893,Annual Mammograms May Have More False-Positives,"October 18, 2011",This article reports on the results of a study...,,While the financial costs of screening mammogr...,,mixture,"Screening,WebMD,women's health"
2,11358,SBRT Offers Prostate Cancer Patients High Canc...,"September 28, 2016",This news release describes five-year outcomes...,"Mary Chris Jaklevic,Steven J. Atlas, MD, MPH,K...",The news release quotes lead researcher Robert...,https://www.healthnewsreview.org/wp-content/up...,mixture,"Association/Society news release,Cancer"
3,10166,"Study: Vaccine for Breast, Ovarian Cancer Has ...","November 8, 2011","While the story does many things well, the ove...",,"The story does discuss costs, but the framing ...",http://clinicaltrials.gov/ct2/results?term=can...,true,"Cancer,WebMD,women's health"
4,11276,Some appendicitis cases may not require ’emerg...,"September 20, 2010",We really don’t understand why only a handful ...,,"""Although the story didn’t cite the cost of ap...",,true,


In [ ]:
print(df.columns)

Index(['claim_id', 'claim', 'date_published', 'explanation', 'fact_checkers',
       'main_text', 'sources', 'label', 'subjects'],
      dtype='object')


In [ ]:
df_clean = df[['claim', 'label', 'explanation']].copy()
df_clean.rename(columns={'explanation': 'evidence'}, inplace=True)

Clean Data


In [ ]:
df_clean = df_clean.dropna()

df_clean = df_clean[df_clean['claim'] != '']
df_clean = df_clean[df_clean['label'] != '']

df_clean.reset_index(drop=True, inplace=True)

print(df_clean.shape)

(9805, 3)


In [ ]:
df_clean.to_csv("clean_train.csv", index=False)

In [ ]:
test_path = os.path.join(path, 'test.tsv')

df_test = pd.read_csv(test_path, sep='\t')

df_test_clean = df_test[['claim', 'label', 'explanation']].copy()
df_test_clean.rename(columns={'explanation': 'evidence'}, inplace=True)

df_test_clean = df_test_clean.dropna()
df_test_clean.reset_index(drop=True, inplace=True)

df_test_clean.to_csv("clean_test.csv", index=False)

In [ ]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_csv("clean_train.csv")
df.head()

,claim,label,evidence
0,"""The money the Clinton Foundation took from fr...",false,"""Gingrich said the Clinton Foundation """"took m..."
1,Annual Mammograms May Have More False-Positives,mixture,This article reports on the results of a study...
2,SBRT Offers Prostate Cancer Patients High Canc...,mixture,This news release describes five-year outcomes...
3,"Study: Vaccine for Breast, Ovarian Cancer Has ...",true,"While the story does many things well, the ove..."
4,Some appendicitis cases may not require ’emerg...,true,We really don’t understand why only a handful ...


In [ ]:
df['text'] = df['claim'] + " " + df['evidence']
texts = df['text'].tolist()

Load embedding model

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings = model.encode(texts, show_progress_bar=True)

Batches:   0%|          | 0/307 [00:00<?, ?it/s]

Store in vector database

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

Build retrieval function

In [ ]:
def retrieve(query, top_k=3):
    query_vec = model.encode([query])

    distances, indices = index.search(query_vec, top_k)

    results = []
    for idx in indices[0]:
        results.append(df.iloc[idx]['text'])

    return results

Test retrieval

In [ ]:
query = "Vaccines cause autism"

results = retrieve(query)

for i, res in enumerate(results):
    print(f"\nResult {i+1}:\n", res)


Result 1:
 Still no autism-vaccine link, say health officials. Federal health officials said on  Thursday the government has not conceded that vaccines cause  autism even after a Georgia girl won federal compensation in a  case arguing a vaccine led to her brain damage.

Result 2:
 Data suppressed by the CDC proved that the MMR vaccine produces a 340% increased risk of autism in African-American boys. The claim  put forward in a video is that earlier MMR vaccination is associated with an increased risk of autism in African-American boys and that the CDC has spent the last 13 years covering this linkage up.

Result 3:
 As the number of vaccines administered to children has increased in the United States, so has the autism rate. A viral meme from NaturalNews.com uses dates, rising numbers in vaccines given to babies, and rising rates of autism to suggest a relationship between vaccines and autism. The meme is inflating the number of vaccines, which stood at 10 in 2008 and 2013, accordin

# **VADER MODEL**

In [ ]:
!pip install nltk


In [ ]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

In [ ]:
def get_sentiment(text):
    score = sia.polarity_scores(text)

    compound = score['compound']

    if compound >= 0.05:
        label = "Positive"
    elif compound <= -0.05:
        label = "Negative"
    else:
        label = "Neutral"

    return label, compound

In [ ]:
sentiments = df['evidence'].apply(get_sentiment)

df['sentiment'] = sentiments.apply(lambda x: x[0])
df['sentiment_score'] = sentiments.apply(lambda x: x[1])

df.head()

,claim,label,evidence,text,sentiment,sentiment_score
0,"""The money the Clinton Foundation took from fr...",false,"""Gingrich said the Clinton Foundation """"took m...","""The money the Clinton Foundation took from fr...",Negative,-0.2983
1,Annual Mammograms May Have More False-Positives,mixture,This article reports on the results of a study...,Annual Mammograms May Have More False-Positive...,Negative,-0.3182
2,SBRT Offers Prostate Cancer Patients High Canc...,mixture,This news release describes five-year outcomes...,SBRT Offers Prostate Cancer Patients High Canc...,Negative,-0.8929
3,"Study: Vaccine for Breast, Ovarian Cancer Has ...",true,"While the story does many things well, the ove...","Study: Vaccine for Breast, Ovarian Cancer Has ...",Negative,-0.2382
4,Some appendicitis cases may not require ’emerg...,true,We really don’t understand why only a handful ...,Some appendicitis cases may not require ’emerg...,Positive,0.1982


In [ ]:
df.to_csv("final_dataset_with_sentiment.csv", index=False)

# **INTEGRATION**

In [ ]:
def split_evidence(evidences):
    pos, neg, neu = [], [], []

    for ev in evidences:
        label, score = get_sentiment(ev)

        if label == "Positive":
            pos.append(ev)
        elif label == "Negative":
            neg.append(ev)
        else:
            neu.append(ev)

    return pos, neg, neu

In [ ]:
claim = "Vaccines cause autism"

retrieved = retrieve(claim)

In [ ]:
pos_ev, neg_ev, neu_ev = split_evidence(retrieved)

In [ ]:
claim = "Vaccines cause autism"

retrieved = retrieve(claim)

pos_ev, neg_ev, neu_ev = split_evidence(retrieved)

print("Positive Evidence:", pos_ev[:2])
print("Negative Evidence:", neg_ev[:2])

Positive Evidence: ['As the number of vaccines administered to children has increased in the United States, so has the autism rate. A viral meme from NaturalNews.com uses dates, rising numbers in vaccines given to babies, and rising rates of autism to suggest a relationship between vaccines and autism. The meme is inflating the number of vaccines, which stood at 10 in 2008 and 2013, according to the CDC. Meanwhile, the rate of children diagnosed with autism is increasing, though the numbers NaturalNews.com offered are not entirely accurate. The problem with this claim, however, is putting the two numbers together to lead people to believe that vaccines are causing the increase in autism rates. A\xa0pediatrician and specialist in infectious diseases said that any connection between vaccines (with or without thimerosal) and autism has been thoroughly dismissed through more than two dozen peer-reviewed studies.']
Negative Evidence: ['Still no autism-vaccine link, say health officials. Fed

In [ ]:
def pro_agent(claim, evidence):
    if not evidence:
        return "No strong supporting evidence found."

    response = "The claim is supported based on the following evidence:\n\n"

    for ev in evidence.split("\n"):
        response += f"- {ev}\n"

    response += "\nThese points suggest the claim may be true."

    return response

In [ ]:
def con_agent(claim, evidence):
    if not evidence:
        return "No strong opposing evidence found."

    response = "The claim is challenged based on the following evidence:\n\n"

    for ev in evidence.split("\n"):
        response += f"- {ev}\n"

    response += "\nThese points suggest the claim may be false."

    return response

In [ ]:
def judge_agent(claim, pos_ev, neg_ev):

    pos_score = len(pos_ev)
    neg_score = len(neg_ev)

    if pos_score > neg_score:
        verdict = "True"
        confidence = int((pos_score / (pos_score + neg_score + 1)) * 100)
    elif neg_score > pos_score:
        verdict = "False"
        confidence = int((neg_score / (pos_score + neg_score + 1)) * 100)
    else:
        verdict = "Uncertain"
        confidence = 50

    explanation = f"""
The decision is based on comparing supporting and opposing evidence.

Supporting evidence count: {pos_score}
Opposing evidence count: {neg_score}
"""

    return {
        "verdict": verdict,
        "confidence": confidence,
        "reason": explanation
    }

In [ ]:
def full_pipeline(claim):

    retrieved = retrieve(claim)

    pos_ev, neg_ev, neu_ev = split_evidence(retrieved)

    pro = pro_agent(claim, "\n".join(pos_ev))
    con = con_agent(claim, "\n".join(neg_ev))

    judgment = judge_agent(claim, pos_ev, neg_ev)

    return {
        "claim": claim,
        "pro_argument": pro,
        "con_argument": con,
        "judgment": judgment
    }

In [ ]:
result = full_pipeline("Vaccines cause autism")

print(result["judgment"])

{'verdict': 'Uncertain', 'confidence': 50, 'reason': '\nThe decision is based on comparing supporting and opposing evidence.\n\nSupporting evidence count: 1\nOpposing evidence count: 1\n'}


In [ ]:
def score_evidence(text):
    label, score = get_sentiment(text)

    length_factor = min(len(text) / 200, 1)   # normalize length
    final_score = abs(score) * length_factor

    return final_score, label

In [ ]:
def filter_strong_evidence(evidences, threshold=0.2):
    strong = []

    for ev in evidences:
        score, label = score_evidence(ev)

        if score >= threshold:
            strong.append((ev, score))

    # sort by strength
    strong.sort(key=lambda x: x[1], reverse=True)

    return [ev[0] for ev in strong[:5]]  # top 5

In [ ]:
def pro_agent(claim, evidence_list):

    strong_evidence = filter_strong_evidence(evidence_list)

    if not strong_evidence:
        return {
            "argument": "Weak support: No strong positive evidence found.",
            "score": 0
        }

    total_score = sum([score_evidence(ev)[0] for ev in strong_evidence])

    argument = "Supporting Evidence:\n"

    for ev in strong_evidence:
        argument += f"- {ev}\n"

    argument += "\nConclusion: Evidence leans toward the claim being TRUE."

    return {
        "argument": argument,
        "score": total_score
    }

In [ ]:
def con_agent(claim, evidence_list):

    strong_evidence = filter_strong_evidence(evidence_list)

    if not strong_evidence:
        return {
            "argument": "Weak opposition: No strong negative evidence found.",
            "score": 0
        }

    total_score = sum([score_evidence(ev)[0] for ev in strong_evidence])

    argument = "Opposing Evidence:\n"

    for ev in strong_evidence:
        argument += f"- {ev}\n"

    argument += "\nConclusion: Evidence suggests the claim may be FALSE."

    return {
        "argument": argument,
        "score": total_score
    }

In [ ]:
def judge_agent(claim, pro_output, con_output):

    pro_score = pro_output["score"]
    con_score = con_output["score"]

    total = pro_score + con_score + 1e-6

    if pro_score > con_score:
        verdict = "True"
    elif con_score > pro_score:
        verdict = "False"
    else:
        verdict = "Uncertain"

    confidence = int((max(pro_score, con_score) / total) * 100)

    explanation = f"""
Decision based on weighted evidence scoring.

Pro Score: {pro_score:.2f}
Con Score: {con_score:.2f}

Higher score indicates stronger supporting evidence.
"""

    return {
        "verdict": verdict,
        "confidence": confidence,
        "reason": explanation
    }

In [ ]:
def full_pipeline(claim):

    retrieved = retrieve(claim)

    pos_ev, neg_ev, neu_ev = split_evidence(retrieved)

    pro = pro_agent(claim, pos_ev)
    con = con_agent(claim, neg_ev)

    judgment = judge_agent(claim, pro, con)

    return {
        "claim": claim,
        "pro": pro,
        "con": con,
        "judgment": judgment
    }

In [ ]:
result = full_pipeline("cold cause fever")

print(result["pro"]["argument"])
print(result["con"]["argument"])
print(result["judgment"])

Supporting Evidence:
- Facebook post Says for otherwise healthy people “experiencing mild to moderate respiratory symptoms with or without a COVID-19 diagnosis … only high temperatures kill a virus, so let your fever run high,” but not over 103 or 104 degrees. A fever makes it harder for some viruses to survive, but it’s not yet known whether that’s true for the novel coronavirus. You might not need to treat a fever that’s under 103 degrees. But it depends on age, general health, other symptoms and other factors. If you have COVID-19 symptoms such as fever, cough and shortness of breath, and think you have been exposed to COVID-19, call your healthcare provider.

Conclusion: Evidence leans toward the claim being TRUE.
Opposing Evidence:
- This new virus is not heat-resistant and will be killed by a temperature of just 26/27 degrees. It hates the Sun. There’s no evidence for this. There’s evidence that similar viruses transmit less well in the heat, but many countries with reported Covi